# 🏎️ F1 Lap Time Predictor
## Notebook 2 — Model Analysis & SHAP

> Run `python src/train.py` before opening this notebook.

1. Load models + evaluate on 2024 test set
2. Model comparison
3. SHAP feature importance
4. Tyre degradation vs SHAP
5. Driver residual analysis
6. Pit strategy classifier analysis

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('dark_background')
ACCENT='#58a6ff'; DANGER='#f85149'; SUCCESS='#3fb950'; YELLOW='#f0a500'
COMPOUND_COLORS={'SOFT':'#f85149','MEDIUM':'#f0a500','HARD':'#e6edf3','INTER':'#3fb950','WET':'#58a6ff'}

MODELS_DIR = '../models'
lap_model = joblib.load(f'{MODELS_DIR}/xgb_lap_model.pkl')
prep      = joblib.load(f'{MODELS_DIR}/lap_preprocessor.pkl')
fnames    = joblib.load(f'{MODELS_DIR}/feature_names.pkl')

with open(f'{MODELS_DIR}/training_summary.json') as f:
    summary = json.load(f)

print('Models loaded ✅')
pd.DataFrame(summary['lap_time_results'])

In [ ]:
from src.data_pipeline import load_and_split
splits = load_and_split()
X_train, X_test, y_train, y_test = splits['main']
full_df = splits['full_df']

X_test_t = prep.transform(X_test)
y_pred   = lap_model.predict(X_test_t)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'Test MAE:  {mae*1000:.1f} ms')
print(f'Test RMSE: {rmse*1000:.1f} ms')
print(f'Test R²:   {r2:.4f}')

In [ ]:
# Actual vs predicted scatter
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for ax in axes: ax.set_facecolor('#161b22')

axes[0].scatter(y_test[:1000], y_pred[:1000], alpha=0.3, s=4, color=ACCENT)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, '--', color='#8b949e', linewidth=1, label='Perfect prediction')
axes[0].set_xlabel('Actual Lap Time (s)')
axes[0].set_ylabel('Predicted Lap Time (s)')
axes[0].set_title(f'Actual vs Predicted (R²={r2:.4f})', fontsize=11)
axes[0].legend()

residuals = (y_test - y_pred) * 1000
axes[1].hist(residuals, bins=60, color=ACCENT, alpha=0.8, edgecolor='none')
axes[1].axvline(0, color=DANGER, linewidth=1.5, linestyle='--', label='Zero error')
axes[1].axvline(residuals.mean(), color=YELLOW, linewidth=1.2, linestyle=':', label=f'Mean: {residuals.mean():.0f}ms')
axes[1].set_xlabel('Residual (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual Distribution (MAE={mae*1000:.0f}ms)', fontsize=11)
axes[1].legend(fontsize=9)

for ax in axes: ax.grid(alpha=0.12)
plt.tight_layout()
plt.savefig('../outputs/model_fit.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# SHAP values
print('Computing SHAP (TreeExplainer)...')
explainer   = shap.TreeExplainer(lap_model)
X_sample    = X_test_t[:800]
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list): shap_values = shap_values[1]
print(f'SHAP shape: {shap_values.shape}')

mean_sv = np.abs(shap_values).mean(axis=0)
top20   = np.argsort(mean_sv)[-20:]

print('\nTop 10 most important features:')
for i in np.argsort(mean_sv)[-10:][::-1]:
    print(f'  {fnames[i]:<35} {mean_sv[i]:.4f}s')

In [ ]:
# Beeswarm
fig, ax = plt.subplots(figsize=(11,8))
ax.set_facecolor('#161b22')

fn_top = [fnames[i] for i in top20]
sv_top = shap_values[:, top20]

for row_i, (sv_col, fname) in enumerate(zip(sv_top.T, fn_top)):
    jitter = np.random.normal(0, 0.09, len(sv_col))
    c = [DANGER if v > 0 else SUCCESS for v in sv_col]
    ax.scatter(sv_col, row_i + jitter, c=c, alpha=0.3, s=7)

ax.set_yticks(range(20))
ax.set_yticklabels(fn_top, fontsize=9)
ax.axvline(0, color='#8b949e', linewidth=0.8, linestyle='--')
ax.set_xlabel('SHAP value (seconds added to lap time)')
ax.set_title('SHAP Beeswarm — Top 20 Features', fontsize=12)
ax.grid(axis='x', alpha=0.12)
slow_p = mpatches.Patch(color=DANGER,  label='Slower')
fast_p = mpatches.Patch(color=SUCCESS, label='Faster')
ax.legend(handles=[slow_p, fast_p], facecolor='#161b22')
plt.tight_layout()
plt.savefig('../outputs/shap_beeswarm.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# SHAP dependence: tyre life
ti = fnames.index('tyre_life')
tyre_sv = shap_values[:, ti]
tyre_v  = X_sample[:, ti]

fig, ax = plt.subplots(figsize=(9,5))
ax.set_facecolor('#161b22')
sc = ax.scatter(tyre_v, tyre_sv, c=tyre_sv, cmap='RdYlGn_r', s=8, alpha=0.5, vmin=-1, vmax=3)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('SHAP value (s)', color='#e6edf3')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#e6edf3')
ax.axhline(0, color='#8b949e', linewidth=0.8, linestyle='--')
ax.set_xlabel('Tyre Age (laps)')
ax.set_ylabel('SHAP value — seconds added to lap time')
ax.set_title('Tyre Age → Lap Time Impact', fontsize=12)
ax.grid(alpha=0.12)
plt.tight_layout()
plt.savefig('../outputs/shap_tyre_dependence.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# Pit classifier metrics
pit_res = summary.get('pit_results', {})
print('=== Pit Stop Classifier ===')
for k, v in pit_res.items():
    print(f'  {k}: {v}')

# Sector model metrics  
print('\n=== Sector Models ===')
for key, m in summary.get('sector_results', {}).items():
    print(f'  {key}: MAE={m["mae_ms"]}ms, R²={m["r2"]}')

In [ ]:
# Final summary
print('=== PROJECT SUMMARY ===')
print(f'Training seasons:  2022-2023')
print(f'Test season:       2024 (temporal split)')
print(f'Lap time MAE:      {mae*1000:.1f} ms')
print(f'Lap time R²:       {r2:.4f}')
print()
print('Top 5 predictors of fast lap times:')
for i, idx in enumerate(np.argsort(mean_sv)[-5:][::-1], 1):
    print(f'  {i}. {fnames[idx]} ({mean_sv[idx]:.4f}s impact)')